# Portfolio tool: daily stock returns, with and without permutation entropy

[Open in Colab](https://colab.research.google.com/github/Gaire-commits/entropy-gate-pipeline/blob/main/notebooks/portfolio_tool.ipynb)

**What it does.** It forecasts which stocks will do best over the next 21 sessions, using an interpretable ridge model on time-series and cross-sectional features. It holds the best fifth as a long-only book in three staggered tranches and charges realistic trading costs. Then it adds permutation entropy (PE) and three interactions fixed in advance, and tests on the same dates whether they improve anything.

**The rules are frozen** in `protocols/portfolio_v1.yaml`, chosen on development and validation only.

**The hold-out is locked.** This notebook loads nothing after 2024-01-02. Its one scoring, shared with the volatility protocol, is a separate deliberate step in `notebooks/reserved_evaluation.ipynb`.

**How to run.** Runtime → Run all, on a CPU runtime. It needs the Colab Secrets `APCA_API_KEY_ID` and `APCA_API_SECRET_KEY`. The first run fetches about 10 minutes of data; the rest takes a few minutes. Results print here; nothing is downloaded.

**Three separate questions.** Does the structure exist (the information coefficient)? Does it forecast out of sample (Clark-West, against the smaller model)? Does it survive costs (net returns)? A yes to one is not a yes to the next.

This is research on historical data, not investment advice. Nothing here places an order. The logic lives in `src/portfolio.py`, with tests in `tests/test_portfolio.py`.

## 1. Setup

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
REPO = '/content/entropy-gate-pipeline' if IN_COLAB else os.environ.get('PORTFOLIO_REPO', os.getcwd())
if IN_COLAB:
    if os.path.isdir(f'{REPO}/.git'):
        !cd {REPO} && git pull -q --ff-only
    else:
        !git clone -q https://github.com/Gaire-commits/entropy-gate-pipeline.git {REPO}
    %cd {REPO}
    !git log --oneline -1
    from google.colab import userdata
    try:
        with open('.env', 'w') as f:
            f.write(f"APCA_API_KEY_ID={userdata.get('APCA_API_KEY_ID')}\nAPCA_API_SECRET_KEY={userdata.get('APCA_API_SECRET_KEY')}\n")
        print('keys written to a local .env (never printed, never committed)')
    except Exception as exc:
        print('No Alpaca keys in Colab Secrets:', type(exc).__name__, '- add them from the key icon, then run this cell again.')
else:
    os.chdir(REPO)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
sys.path[:0] = [REPO, f'{REPO}/scripts']

from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
from src import portfolio as pf, volpanel, audit
import portfolio_rules as rules
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

## 2. The frozen settings

Read from the protocol, so nothing here can drift from what was registered.

In [ ]:
cfg, protocol = pf.load_protocol('protocols/portfolio_v1.yaml')
LOCK = pf.lock_date(cfg)
START, END = '2016-01-04', '2026-10-13'
REF = os.environ.get('PORTFOLIO_REF', '/content/portfolio_data/reference')
print(cfg)
print('\nregistered', protocol['registered'], '| nothing after', LOCK.date(), 'is loaded here')

## 3. Data

SIP daily bars from Alpaca, for `universe/sp500.csv`, through the project's tested fetcher: `all`-adjusted prices for returns, raw prices and volume for the price and liquidity rules, split-adjusted prices to find adjustment steps, and the corporate-action records. Plus SPY (the market) and BIL (Treasury bills, the risk-free proxy), checked against total returns recorded independently in the protocol. Finished files are kept, so after a disconnect just run the cell again.

In [ ]:
if IN_COLAB and not Path(f'{REF}/corporate_actions.parquet').exists():
    !python scripts/fetch_reference.py --universe universe/sp500.csv --start {START} --end {END} --out {REF} --parts daily,corporate_actions
universe = pd.read_csv('universe/sp500.csv')
m = rules.load_market(REF, cfg)
index = m.close.index
try:
    bench = pf.fetch_benchmarks(os.environ.get('PORTFOLIO_BENCHMARKS', f'{REF}/benchmarks_to_{LOCK.date()}.parquet'), START, str(LOCK.date()))
    guard = pf.check_benchmarks(bench, protocol['benchmark_reference'], m.truncated)
    print('benchmarks against the recorded total returns:', 'PASS' if guard['ok'].all() else 'FAIL', f"(largest gap {(guard['this_series'] - guard['reference']).abs().max():.2%})")
    assert guard['ok'].all(), 'the benchmark series is not the one recorded in the protocol'
except RuntimeError as exc:
    bench = None
    print('NO BENCHMARK DATA:', exc, '- the market and risk-free lines are left out, and Sharpe and Sortino are on raw returns.')
print(f'{len(index):,} sessions, {index[0].date()} to {index[-1].date()}; {m.close.shape[1]} symbols with bars by then')
print('universe list fetched:', universe['fetched_at'].iloc[0][:10], '| truncated vendor days (no usable price):', [str(d.date()) for d in m.truncated])
print('event sessions (return set to zero):', f'{int(m.event.sum().sum()):,}', '| real bars:', f'{int(m.real.sum().sum()):,}')
big = pf.clean_returns(m).stack().rename('log_return').reset_index()
big.columns = ['date', 'symbol', 'log_return']
print('\nlargest daily moves in the data: read them, they should be real events')
print(big.reindex(big['log_return'].abs().sort_values(ascending=False).index).head(12).to_string(index=False))

## 4. What this data cannot do

**These limits are in every number below.**

| Limit | Effect | What is done about it |
|---|---|---|
| **Survivorship.** The universe is today's S&P 500 list; companies that left the index or failed are missing. | Absolute returns are inflated, by an unknown amount. Beating SPY means little, because SPY held the losers too. | Each stock counts only from its index-join date (Wikipedia, retrieved 2026-11-02), which removes the run-up into the index: equal-weight's 2019-2021 return falls about 3 points a year. Removed companies stay absent; that needs point-in-time data such as CRSP. **Judge the portfolios against equal-weight**, which shares the bias, and read the IC first. |
| **Sectors** are today's labels applied to the whole history. | Mild look-ahead in the sector-relative features. | None; stated here. |
| **The market benchmark** is SPY, weighted by size. | Differs from these portfolios in weighting as well as selection. | Reported, along with equal-weight. |
| **Costs are assumed:** 5 bp one-way plus square-root impact on $10M. | A real book's costs may differ. | That charged 8.4 bp per dollar traded; the published average for live institutional trades is about 9 bp (Frazzini, Israel, Moskowitz). Daily-bar spread estimators could not resolve spreads this small. Section 10 runs 0 to 20 bp. |
| **Execution** assumes a fill at the next session's official close. The vendor's close equals the official close on 99.99% of sampled symbol-days. | Capacity at the close is not modelled beyond the impact term. | The largest trade as a share of daily volume is reported. Section 10 shows the next open, but the vendor's open is the first trade, not the opening auction. |
| **Volume** includes extended hours, about 28% above the session's. | The liquidity rule and impact are slightly lenient. | Stated here. |
| **Corporate actions.** `all` adjustment misses a few events, and records begin late for some kinds. | A missed event would look like a huge return. | Sessions named by a record, a vendor adjustment step or the checked registry earn zero (about 0.01% of bars). Section 3 lists the largest moves left. |
| **Share classes** (GOOG and GOOGL, NWS and NWSA, FOX and FOXA) count as separate stocks. | The book can hold nearly one position twice. | None; stated here. |
| **Selection.** The rules are the best of 24 candidates seen on these windows. | Results on these windows are flattered. | Disclosed in the protocol. The IC is less exposed than the book, since the forecast's form was never searched. |
| **Prior exposure.** These dates were studied in earlier work (`protocols/reserved_period_disclosure.md`). | Walk-forward stops the model seeing the future; it cannot stop the researcher's choices being informed by it. | Every window is "reserved for this specification but previously exposed through related research". |
| **Sample size.** About 36 decisions a year, overlapping, and one market history. | Intervals are wide; a null may only mean too little power. | Intervals are reported with every comparison. |

## 5. How look-ahead is prevented, then checked on the real data

- **Features and eligibility** at t use data through the close of t. Liquidity uses earlier sessions only. Price and volume rules use raw prices, so later adjustments can't leak in.
- **Execution:** a decision at the close of t fills at the close of t+1, so that session's return is not earned.
- **The target** runs from that fill to the fill 21 sessions later, and is empty where no fill can happen.
- **Training** uses only rows whose target was fully realized by the refit date.
- **Costs and impact** use volatility and volume known at t.

Two checks: scramble every price after a date and confirm that nothing a decision on that date sees has changed; and reconcile each tranche's actual return with the target the model was fitted to.

In [ ]:
target = pf.forward_returns(m, cfg)
feats, eligible = pf.build_features(m, cfg)
check_day = index[252 + cfg.min_train_sessions + 250]
check = pf.no_lookahead_check(m, cfg, check_day)
print('scrambling every price after', check_day.date(), '-> unchanged at that date:', check)
assert all(check.values()), 'LOOK-AHEAD: stop here'

dates = pf.decision_dates(index, cfg)
end = pf.scored_end(index, cfg)
print(f'\ndecisions: {len(dates)}, {dates[0].date()} to {dates[-1].date()}, every {pf.step(cfg)} sessions; returns scored to {end.date()}')
print('eligible stocks at a decision, median by year:')
print(eligible.loc[dates].sum(axis=1).groupby(dates.year).median().astype(int).to_frame('eligible').T.to_string())

## 6. The features

All at the close of t.

| Group | Features | Meaning |
|---|---|---|
| Time series | `ts_ret_1`, `ts_ret_5`, `ts_ret_21`, `ts_mom_12_1`, `ts_vol_ratio` | The stock against its own history: lagged returns and 12-1 momentum in units of its own volatility, and short over long volatility |
| Cross-section | `xs_ret_5`, `xs_ret_21`, `xs_mom_6_1`, `xs_mom_12_1`, `xs_vol_63` | Its rank among eligible stocks that day |
| Sector | `sec_rel_ret_21`, `sec_rel_mom_12_1`, `sector_ret_21` | Its rank within its sector, and its sector's rank among sectors |
| PE (added later) | `xs_pe` | Rank of permutation entropy of daily returns: order 3, delay 1, 63 patterns |
| Interactions (added later) | `pe_x_mom_12_1`, `pe_x_ret_5`, `pe_x_vol_63` | Does disorder change how momentum or the 5-day move carries on? The volatility term is the control, since entropy moves with volatility |

In [ ]:
sample = dates[::10]
stack = pd.DataFrame({n: feats[n].loc[sample].stack() for n in pf.BASE + pf.PE_FEATURE})
print('how PE relates to the other features (rank correlation, sampled decisions):')
print(stack.corr(method='spearman')['xs_pe'].drop('xs_pe').sort_values().to_frame('with xs_pe').T.to_string())

## 7. The model without PE

Ridge regression, refit every quarter on all earlier realized data, predicting each stock's rank in the next 21 sessions' return. It uses one fixed light penalty rather than a tuned one, so there is nothing to overfit in choosing it. A coefficient reads as: the change in forecast rank per one standard deviation of the feature.

In [ ]:
results = {'base': pf.run_model('base', feats, eligible, target, m, dates, cfg, end)}
print(pf.coefficient_table(results['base']['coefs']).to_string())
print(f"\n{len(results['base']['coefs'])} refits. same_sign: how often a coefficient keeps its average sign. One that flips is not something to lean on.")
print('Read related features together: ts_ret_21 and xs_ret_21 carry much of the same information, so opposite signs partly cancel.')
print('\nthe target against the book, per tranche:')
for k, tr in enumerate(pf.tranche_targets(results['base']['forecasts'], m, cfg)):
    c = pf.reconcile_target(tr, m, cfg, end)
    print(f"  tranche {k}: {len(c)} holding periods, largest gap {c['gap'].abs().max():.1e}, names with no fill {int(c['target'].isna().sum())}; a target one session early would be off by {(c['book'] - c['unlagged']).abs().median() * 1e4:.0f} bp")

## 8. The portfolio against the benchmarks

**The rules.** Three staggered tranches, each held 21 sessions, so one turns over every 7 sessions and no single calendar phase decides the result. Each tranche holds the best 20% of eligible stocks by forecast, equally weighted, keeps a holding while it stays in the best 40%, and allows no sector above 30% of its names. A stock with no bar that session can't be traded and stays at its drifted weight.

**The benchmarks.** Equal-weight is every eligible stock on the same schedule at the same costs. Market is SPY held throughout. Sharpe and Sortino are measured over BIL.

In [ ]:
ew = pf.simulate(pf.equal_weight_targets(eligible, dates), m, cfg, end)
rf = pf.benchmark_returns(bench, 'BIL', cfg.execution, m.truncated) if bench is not None else None
market = pf.benchmark_returns(bench, 'SPY', cfg.execution, m.truncated) if bench is not None else None
def table(names):
    series = {n: results[n]['daily'] for n in names}
    series['equal-weight'] = ew
    if market is not None:
        series['market (SPY)'] = market
    return pf.scorecard(series, rf, cfg, {n: results[n]['ic'] for n in names})
print(table(['base']).to_string(index=False))
d = results['base']['daily']
print(f"\nnames held, median: {d['names'].median():.0f}; cost per dollar traded {(d['cost'].sum() / (2 * d['turnover'].sum())) * 1e4:.1f} bp; total cost {d['cost'].sum():.1%} of capital; largest trade {d['max_participation'].max():.1%} of a stock's daily dollar volume at ${cfg.aum:,.0f}")

In [ ]:
import matplotlib.pyplot as plt
def plot(names):
    fig, ax = plt.subplots(figsize=(10, 4))
    for n in names:
        ax.plot((1 + results[n]['daily']['net']).cumprod(), label=f'{n}, net')
    ax.plot((1 + ew['net']).cumprod(), label='equal-weight, net', color='black', lw=1)
    if market is not None:
        ax.plot((1 + market.reindex(ew.index).fillna(0)).cumprod(), label='market (SPY)', color='grey', lw=1)
    ax.axvline(pd.Timestamp(cfg.validation_start), color='grey', ls=':')
    ax.set_yscale('log'); ax.set_ylabel('growth of 1 (log scale)'); ax.legend(); ax.set_title('Survivor-biased universe: compare the lines, not their level')
    plt.show()
plot(['base'])

## 9. Does PE add anything?

**The rule, fixed before any result, for each larger model against `base`:**

- **Forecast:** the Clark-West t is at least 1.645 (one-sided 5%) **and** the 95% interval for the IC gap lies above zero.
- **Portfolio:** the 95% interval for the net-return gap lies above zero.

`base_pe` is the one primary test. `base_pe_x` is secondary. Rows, folds and portfolio rules are identical across models. On these windows the result is context only: the verdict that counts comes from the reserved window, once.

In [ ]:
for name in ('base_pe', 'base_pe_x'):
    results[name] = pf.run_model(name, feats, eligible, target, m, dates, cfg, end)
print(table(['base', 'base_pe', 'base_pe_x']).to_string(index=False))
plot(['base', 'base_pe', 'base_pe_x'])

In [ ]:
spans = {'development': (None, pd.Timestamp(cfg.validation_start) - pd.Timedelta(days=1)), 'validation': (pd.Timestamp(cfg.validation_start), None)}
rows = []
for name in ('base_pe', 'base_pe_x'):
    for span, (a, b) in spans.items():
        c = pf.paired_comparison(results['base'], results[name], target, eligible, rf, a, b, cfg.seed)
        rows.append({'model': name, 'window': span, **{k: c[k] for k in ('dates', 'ic_small', 'ic_large', 'ic_gap', 'ic_gap_lo', 'ic_gap_hi', 'clark_west_t',
                     'net_return_gap_ann', 'net_gap_lo', 'net_gap_hi', 'sharpe_gap', 'sharpe_gap_lo', 'sharpe_gap_hi')}, 'verdict': pf.verdict(c)})
print(pd.DataFrame(rows).drop(columns='verdict').to_string(index=False))
print()
for r in rows:
    print(f"{r['model']:10s} {r['window']:12s} {r['verdict']}")
print('\nWhat the PE terms look like inside the largest model:')
print(pf.coefficient_table(results['base_pe_x']['coefs']).loc[pf.PE_FEATURE + pf.INTERACTIONS].to_string())

## 10. Sensitivity: the same forecasts under other costs and execution

These forecasts are not re-chosen on this table, so it can't select a better model; it shows how fragile each result is. The next-open row keeps forecasts trained for the close-to-close target.

In [ ]:
variants = {'0 bp': replace(cfg, cost_bps=0, impact=0), '5 bp, no impact': replace(cfg, impact=0), 'as frozen': cfg, '10 bp': replace(cfg, cost_bps=10),
            '20 bp': replace(cfg, cost_bps=20), '$100M': replace(cfg, aum=100e6), 'next open': replace(cfg, execution='next_open')}
rows = []
start = pd.Timestamp(cfg.validation_start)
for label, c in variants.items():
    for name in results:
        p = pf.performance(pf.resimulate(results[name], m, c)['net'].loc[start:], rf)
        rows.append({'variant': label, 'model': name, 'return_ann': p['return_ann'], 'sharpe': p['sharpe']})
    p = pf.performance(pf.simulate(pf.equal_weight_targets(eligible, dates), m, c, end)['net'].loc[start:], rf)
    rows.append({'variant': label, 'model': 'equal-weight', 'return_ann': p['return_ann'], 'sharpe': p['sharpe']})
s = pd.DataFrame(rows)
print('Validation, net:')
print(s.pivot(index='variant', columns='model', values='return_ann').reindex(variants).to_string())
print()
print(s.pivot(index='variant', columns='model', values='sharpe').reindex(variants).to_string())

## 11. The tool: the next tranche it would buy

The PE-free model's next purchase as of the last session this notebook may load (2024-01-02, the lock), with the features behind each pick. Orders from that date would fill at the next session's close. It shows how the tool reads, not a current recommendation. `why` lists each stock's two largest contributions to its forecast, relative to the other stocks that day.

In [ ]:
per = pf.tranche_targets(results['base']['forecasts'], m, cfg)
due = per[len(dates) % cfg.tranches]
now = pf.current_portfolio('base', feats, eligible, target, m, cfg, set(due[max(due)]), fit_until=end)
print(f"data through {now['date'].iloc[0].date()}, {len(now)} names in the tranche. Sector shares:", now['sector'].value_counts(normalize=True).round(2).to_dict())
print(now.drop(columns='date').head(25).to_string(index=False))